In [1]:
import os
import glob
import json
import pandas as pd
import numpy as np
import time
import warnings
from tqdm import tqdm
from konlpy.tag import Mecab
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.feature_extraction.text")

# =========================================================
# [설정] 실험 조건
# =========================================================
DATA_DIRS = ["data/대법원_판례", "data/하위법원_판례"] 
OUTPUT_DIR = "benchmark_hybrid_results"  # ⭐ 새 결과 폴더
TOP_N_CLASSES = 10
NUM_ITERATIONS = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f">>> 결과 저장 폴더: {OUTPUT_DIR}")

mecab = Mecab()

def pre_tokenize(text):
    """명사만 추출하여 공백으로 구분된 문자열 생성"""
    if not text: return ""
    return " ".join([word for word, pos in mecab.pos(text) if pos.startswith('N')])

def load_and_preprocess_data(data_dirs_list, top_n=None):
    all_files = []
    for d in data_dirs_list:
        files = glob.glob(os.path.join(d, "**", "*.json"), recursive=True)
        all_files.extend(files)
    
    texts = []
    labels = []
    
    print(f">>> 총 {len(all_files)}개 JSON 파일 발견")
    for file_path in tqdm(all_files, desc="JSON Loading"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                content = data.get('판례내용', "").replace("<br/>", " ").replace("\n", " ")
                label = data.get('사건명', "")
                if content and label:
                    texts.append(content)
                    labels.append(label)
        except: 
            continue
    
    df = pd.DataFrame({'text': texts, 'label': labels})
    print(f">>> 로딩 완료: {len(df)}개 문서")
    
    if top_n:
        target_classes = df['label'].value_counts().head(top_n).index.tolist()
        df = df[df['label'].isin(target_classes)].copy()
        print(f">>> Top {top_n} 클래스 필터링: {len(df)}개 문서")
    
    tqdm.pandas(desc="Pre-tokenizing (Mecab)")
    df['tokenized_text'] = df['text'].progress_apply(pre_tokenize)
    
    return train_test_split(
        df['tokenized_text'], df['label'], 
        test_size=0.2, random_state=42, stratify=df['label']
    )

def load_stopwords_from_txt(filepath):
    """TXT 파일에서 불용어 로드"""
    if not os.path.exists(filepath):
        return None
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            words = [line.strip() for line in f if line.strip()]
        return set(words)
    except Exception as e:
        print(f"  ⚠️  Error loading {filepath}: {e}")
        return None

print(f"\n{'='*70}")
print(f"  Hybrid Stopwords Testing: General + Conservative Combinations")
print(f"{'='*70}\n")

print(f">>> 1. 데이터셋 로딩 및 사전 토큰화 시작...")
X_train, X_test, y_train, y_test = load_and_preprocess_data(DATA_DIRS, TOP_N_CLASSES)

# =========================================================
# 불용어 로드
# =========================================================
print(f"\n>>> 2. 불용어 리스트 로드 중...")

# General stopwords
GENERAL_STOPWORD_FILE = "stopwords.txt"
if not os.path.exists(GENERAL_STOPWORD_FILE):
    print(f"  ⚠️  {GENERAL_STOPWORD_FILE} 생성 중...")
    with open(GENERAL_STOPWORD_FILE, "w", encoding="utf-8") as f:
        default_stopwords = ["이", "그", "저", "것", "수", "등", "들", "및", 
                             "에", "의", "를", "가", "은", "는", "을", "와", "과"]
        f.write("\n".join(default_stopwords))

set_general = load_stopwords_from_txt(GENERAL_STOPWORD_FILE)
print(f"  ✅ General stopwords: {len(set_general)}개")

# Conservative stopwords
conservative_path = "improved_stopwords/V6_Conservative.txt"
set_conservative = load_stopwords_from_txt(conservative_path)
if set_conservative:
    print(f"  ✅ Conservative stopwords: {len(set_conservative)}개")
else:
    print(f"  ❌ Conservative 파일을 찾을 수 없습니다: {conservative_path}")
    print(f"  먼저 improve_stopwords.py를 실행하세요.")
    exit(1)

# =========================================================
# 하이브리드 조합 생성
# =========================================================
print(f"\n>>> 3. 하이브리드 조합 생성 중...")

# 교집합 & 합집합
set_intersection = set_general.intersection(set_conservative)
set_union = set_general.union(set_conservative)

# 차집합 (어느 한쪽에만 있는 것들)
only_general = set_general - set_conservative
only_conservative = set_conservative - set_general

print(f"\n  📊 집합 분석:")
print(f"     General:                  {len(set_general):>4}개")
print(f"     Conservative:             {len(set_conservative):>4}개")
print(f"     General ∩ Conservative:   {len(set_intersection):>4}개")
print(f"     General ∪ Conservative:   {len(set_union):>4}개")
print(f"     Only in General:          {len(only_general):>4}개")
print(f"     Only in Conservative:     {len(only_conservative):>4}개")

if only_general:
    print(f"\n  🔍 General에만 있는 단어 (샘플): {list(only_general)[:20]}")
if only_conservative:
    print(f"  🔍 Conservative에만 있는 단어 (샘플): {list(only_conservative)[:20]}")

# =========================================================
# 실험 시나리오 구성
# =========================================================
stopword_sets = {
    "0. Baseline (None)": set(),
    "1. General Only": set_general,
    "2. Conservative Only": set_conservative,
    "3. General ∩ Conservative": set_intersection,
    "4. General ∪ Conservative": set_union,  # ⭐ 주요 테스트
}

# 추가 실험: 단계적 조합
# General + Conservative의 일부만
if only_conservative:
    # Conservative에만 있는 것 중 상위 절반만 추가
    half_conservative = set(list(only_conservative)[:len(only_conservative)//2])
    stopword_sets["5. General + 50% Conservative"] = set_general.union(half_conservative)
    
    # Conservative에만 있는 것 중 상위 25%만 추가
    quarter_conservative = set(list(only_conservative)[:len(only_conservative)//4])
    stopword_sets["6. General + 25% Conservative"] = set_general.union(quarter_conservative)

# 추가 실험: General에서 Conservative를 뺀 것
stopword_sets["7. General - Conservative"] = set_general - set_conservative

# 추가 실험: Conservative에서 General을 뺀 것
stopword_sets["8. Conservative - General"] = set_conservative - set_general

print(f"\n>>> 총 {len(stopword_sets)}개 실험 시나리오 준비 완료")

# =========================================================
# 벤치마크 실행
# =========================================================
print(f"\n{'='*110}")
print(f"  분류 성능 벤치마크 시작 (Top {TOP_N_CLASSES} 클래스, {NUM_ITERATIONS}회 반복)")
print(f"{'='*110}\n")

results = []
print("-" * 110)
print(f"{'Experiment':<35} | {'Acc':<8} | {'F1':<8} | {'Avg Time':<12} | {'Count':<6}")
print("-" * 110)

for name, stop_set in stopword_sets.items():
    stop_list = list(stop_set) if stop_set else None
    times, accuracies, f1s = [], [], []
    
    for iteration in range(NUM_ITERATIONS):
        pipeline = Pipeline([
            ('vect', TfidfVectorizer(
                stop_words=stop_list, 
                max_features=5000,
                tokenizer=None,
                token_pattern=r"(?u)\b\w\w+\b"
            )),
            ('clf', LogisticRegression(C=5.0, max_iter=1000, n_jobs=-1)) 
        ])
        
        start_time = time.time()
        try:
            pipeline.fit(X_train, y_train)
            pred = pipeline.predict(X_test)
            elapsed = time.time() - start_time
            
            times.append(elapsed)
            accuracies.append(accuracy_score(y_test, pred))
            f1s.append(f1_score(y_test, pred, average='weighted'))
            
        except Exception as e:
            print(f"{name:<35} | Error: {e}")
            break
    
    if times:
        avg_acc = np.mean(accuracies)
        avg_f1 = np.mean(f1s)
        avg_time = np.mean(times)
        std_time = np.std(times)
        
        print(f"{name:<35} | {avg_acc:.4f}   | {avg_f1:.4f}   | {avg_time:.3f}s ±{std_time:.2f} | {len(stop_set):<6}")
        
        results.append({
            'Experiment': name,
            'Count': len(stop_set),
            'Accuracy': avg_acc,
            'F1_Score': avg_f1,
            'Avg_Time': avg_time,
            'Std_Time': std_time
        })

print("-" * 110)

# =========================================================
# 결과 저장 및 분석
# =========================================================
if results:
    final_df = pd.DataFrame(results).sort_values(
        by=['F1_Score', 'Accuracy', 'Avg_Time'], 
        ascending=[False, False, True]
    ).round(4)
    
    final_df.insert(0, 'Rank', range(1, len(final_df) + 1))
    
    print("\n" + "="*90)
    print(f"  📊 하이브리드 조합 성능 순위 (Top {TOP_N_CLASSES} 클래스 분류)")
    print("="*90)
    print(final_df.to_string(index=False))
    print("="*90)
    
    # CSV 저장
    csv_path = os.path.join(OUTPUT_DIR, f"benchmark_hybrid_top{TOP_N_CLASSES}.csv")
    final_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 전체 결과 저장: {csv_path}")
    
    # ⭐ 핵심 비교: General vs Union
    print(f"\n{'='*90}")
    print(f"  🎯 핵심 결과 비교")
    print(f"{'='*90}")
    
    general_result = final_df[final_df['Experiment'] == '1. General Only']
    union_result = final_df[final_df['Experiment'] == '4. General ∪ Conservative']
    
    if not general_result.empty and not union_result.empty:
        gen_f1 = general_result['F1_Score'].values[0]
        gen_time = general_result['Avg_Time'].values[0]
        
        union_f1 = union_result['F1_Score'].values[0]
        union_time = union_result['Avg_Time'].values[0]
        
        f1_diff = (union_f1 - gen_f1) * 100
        time_diff = union_time - gen_time
        
        print(f"\n1️⃣  General Only:")
        print(f"   F1: {gen_f1:.4f}, Time: {gen_time:.2f}s, Count: {len(set_general)}")
        
        print(f"\n2️⃣  General ∪ Conservative:")
        print(f"   F1: {union_f1:.4f}, Time: {union_time:.2f}s, Count: {len(set_union)}")
        
        print(f"\n📈 변화:")
        print(f"   F1 Score: {f1_diff:+.2f}%")
        print(f"   Time:     {time_diff:+.2f}s")
        
        if union_f1 > gen_f1:
            print(f"\n🎉 성공! General ∪ Conservative가 General보다 우수합니다!")
        elif abs(union_f1 - gen_f1) < 0.0005:
            print(f"\n✅ 거의 동일한 성능입니다 (차이 < 0.05%)")
        else:
            print(f"\n⚠️  General이 여전히 더 우수합니다")
    
    # 최고 성능
    print(f"\n{'='*90}")
    print(f"  🏆 최종 추천")
    print(f"{'='*90}")
    
    best = final_df.iloc[0]
    print(f"\n최고 F1 성능: {best['Experiment']}")
    print(f"  - F1: {best['F1_Score']:.4f}")
    print(f"  - Accuracy: {best['Accuracy']:.4f}")
    print(f"  - Time: {best['Avg_Time']:.2f}s")
    print(f"  - Count: {best['Count']}개")
    
    # 성능-속도 균형
    balanced = final_df[(final_df['F1_Score'] >= best['F1_Score'] - 0.001) & 
                        (final_df['Avg_Time'] <= best['Avg_Time'])]
    
    if len(balanced) > 1:
        print(f"\n성능 거의 동일 (±0.1%) & 빠른 옵션들:")
        for idx, row in balanced.iterrows():
            print(f"  - {row['Experiment']}: F1={row['F1_Score']:.4f}, Time={row['Avg_Time']:.2f}s")
    
    # 차집합 분석
    print(f"\n{'='*90}")
    print(f"  🔬 차집합 분석")
    print(f"{'='*90}")
    
    only_gen_result = final_df[final_df['Experiment'] == '7. General - Conservative']
    only_con_result = final_df[final_df['Experiment'] == '8. Conservative - General']
    
    if not only_gen_result.empty:
        print(f"\nGeneral에만 있는 단어들의 영향:")
        print(f"  F1: {only_gen_result['F1_Score'].values[0]:.4f}")
        print(f"  개수: {len(only_general)}개")
        print(f"  샘플: {list(only_general)[:10]}")
    
    if not only_con_result.empty:
        print(f"\nConservative에만 있는 단어들의 영향:")
        print(f"  F1: {only_con_result['F1_Score'].values[0]:.4f}")
        print(f"  개수: {len(only_conservative)}개")
        print(f"  샘플: {list(only_conservative)[:10]}")
    
    print(f"\n{'='*90}\n")

else:
    print("\n❌ 실험 결과가 없습니다.")

print(f"✅ 하이브리드 테스트 완료! 결과 폴더: {OUTPUT_DIR}/")

>>> 결과 저장 폴더: benchmark_hybrid_results

  Hybrid Stopwords Testing: General + Conservative Combinations

>>> 1. 데이터셋 로딩 및 사전 토큰화 시작...
>>> 총 33230개 JSON 파일 발견


JSON Loading: 100%|██████████| 33230/33230 [00:12<00:00, 2738.19it/s]


>>> 로딩 완료: 33230개 문서
>>> Top 10 클래스 필터링: 4763개 문서


Pre-tokenizing (Mecab): 100%|██████████| 4763/4763 [00:18<00:00, 256.80it/s]



>>> 2. 불용어 리스트 로드 중...
  ✅ General stopwords: 600개
  ✅ Conservative stopwords: 75개

>>> 3. 하이브리드 조합 생성 중...

  📊 집합 분석:
     General:                   600개
     Conservative:               75개
     General ∩ Conservative:     40개
     General ∪ Conservative:    635개
     Only in General:           560개
     Only in Conservative:       35개

  🔍 General에만 있는 단어 (샘플): ['반대로 말하자면', '같이', '본대로', '영차', '근거로', '이젠', '그에 따르는', '타다', '바로', '무렵', '저것만큼', '그치지 않다', '흐흐', '동시에', '들', '그러면', '의해서', '이러이러하다', '팔', '어찌됏든']
  🔍 Conservative에만 있는 단어 (샘플): ['사항', '거기', '관련', '만', '늘', '가끔', '또는', '의한', '때때로', '은', '아주', '항상', '있어서', '대로', '바', '그것', '아마', '도', '경우', '의하여']

>>> 총 9개 실험 시나리오 준비 완료

  분류 성능 벤치마크 시작 (Top 10 클래스, 5회 반복)

--------------------------------------------------------------------------------------------------------------
Experiment                          | Acc      | F1       | Avg Time     | Count 
-------------------------------------------------------------------------------

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
General Stopwords Analysis and Improvement
일반 불용어 리스트 분석 및 개선
"""

import pandas as pd
from konlpy.tag import Mecab
from collections import Counter
import os

print("="*70)
print("  General Stopwords Analysis")
print("="*70)

# 원본 불용어 로드
with open('stopwords.txt', 'r', encoding='utf-8') as f:
    general_stopwords = [line.strip() for line in f if line.strip()]

print(f"\n총 {len(general_stopwords)}개 불용어")

# =========================================================
# 1. 법률 핵심 용어 탐지
# =========================================================
CRITICAL_LEGAL_TERMS = {
    '조', '항', '호', '목',  # 법조문 구조
    '법',  # 법률명
    '원고', '피고', '피의자', '피해자',  # 당사자
    '법원', '판사', '검사', '변호사',  # 기관/직책
    '판결', '결정', '명령',  # 법적 결과
    '증거', '증인', '증명',  # 소송 절차
    '소송', '재판', '항소', '상고',  # 절차
    '권리', '의무', '책임',  # 법적 개념
    '손해', '배상', '청구',  # 민사
    '형', '벌금', '징역',  # 형사
}

found_legal_terms = []
for word in general_stopwords:
    if word in CRITICAL_LEGAL_TERMS:
        found_legal_terms.append(word)

print(f"\n🚨 발견된 법률 핵심 용어 ({len(found_legal_terms)}개):")
print(f"   {found_legal_terms}")
print(f"\n   ❌ 이 단어들은 절대 제거하면 안 됩니다!")

# =========================================================
# 2. 의성어/의태어 탐지
# =========================================================
ONOMATOPOEIA = {
    '까악', '꽈당', '끙끙', '끼익', '둥둥', '딩동', '뚝뚝',
    '붕붕', '삐걱', '솨', '쉿', '와르르', '와아', '윙윙',
    '엉엉', '졸졸', '좍좍', '주룩주룩', '콸콸', '쾅쾅', '쿵',
    '탕탕', '툭', '퉤', '팍', '퍽', '펄렁', '허걱', '허허',
    '헉', '헉헉', '헐떡헐떡', '휘익', '휴', '흐흐', '흥'
}

found_onomatopoeia = []
for word in general_stopwords:
    if word in ONOMATOPOEIA:
        found_onomatopoeia.append(word)

print(f"\n💭 의성어/의태어 ({len(found_onomatopoeia)}개):")
print(f"   {found_onomatopoeia}")
print(f"   ℹ️  법률 문서에서 거의 등장하지 않음 (제거해도 무방)")

# =========================================================
# 3. 동사/형용사 (Mecab 분석)
# =========================================================
print(f"\n⏳ Mecab 품사 분석 중...")
mecab = Mecab()

pos_distribution = Counter()
verbs_adjectives = []
particles = []
nouns = []

for word in general_stopwords:
    try:
        pos = mecab.pos(word)
        if pos:
            pos_tag = pos[0][1]
            pos_distribution[pos_tag] += 1
            
            if pos_tag.startswith('V'):  # 동사/형용사
                verbs_adjectives.append(word)
            elif pos_tag.startswith('J'):  # 조사
                particles.append(word)
            elif pos_tag.startswith('N'):  # 명사
                nouns.append(word)
    except:
        pass

print(f"\n📊 품사 분포 (상위 10개):")
for pos, count in pos_distribution.most_common(10):
    print(f"   {pos}: {count}개")

print(f"\n🔍 명사로 태깅된 단어 ({len(nouns)}개):")
print(f"   {nouns[:30]}...")

# =========================================================
# 4. 문제가 있는 단어들
# =========================================================
PROBLEMATIC = {
    # 법률 핵심 용어
    '조', '항', '호', '목', '법',
    
    # 시간/숫자 관련 (법률에서 중요할 수 있음)
    '년', '월', '일', '시각', '시간',
    
    # 일반 명사 (맥락에 따라 중요할 수 있음)
    '결과', '결국', '전부', '전자', '전후',
    '기타', '등', '등등',
}

found_problematic = []
for word in general_stopwords:
    if word in PROBLEMATIC:
        found_problematic.append(word)

print(f"\n⚠️  문제 가능성 있는 단어 ({len(found_problematic)}개):")
print(f"   {found_problematic}")

# =========================================================
# 5. 개선안 생성
# =========================================================
print(f"\n{'='*70}")
print(f"  개선안 생성")
print(f"{'='*70}")

# Version 1: 법률 핵심 용어만 제거
improved_v1 = [w for w in general_stopwords if w not in CRITICAL_LEGAL_TERMS]
print(f"\n✅ V1: 법률 핵심 용어 제거")
print(f"   원본: {len(general_stopwords)}개 → 개선: {len(improved_v1)}개")
print(f"   제거된 단어: {[w for w in general_stopwords if w in CRITICAL_LEGAL_TERMS]}")

# Version 2: 법률 용어 + 의성어 제거
improved_v2 = [w for w in general_stopwords 
               if w not in CRITICAL_LEGAL_TERMS and w not in ONOMATOPOEIA]
print(f"\n✅ V2: 법률 용어 + 의성어 제거")
print(f"   원본: {len(general_stopwords)}개 → 개선: {len(improved_v2)}개")

# Version 3: 법률 용어 + 의성어 + 문제 단어 제거
improved_v3 = [w for w in general_stopwords 
               if w not in PROBLEMATIC and w not in ONOMATOPOEIA]
print(f"\n✅ V3: 법률 용어 + 의성어 + 문제 단어 제거")
print(f"   원본: {len(general_stopwords)}개 → 개선: {len(improved_v3)}개")

# Version 4: 보수적 - 조사와 접속사만 유지
improved_v4 = []
for word in general_stopwords:
    try:
        pos = mecab.pos(word)
        if pos:
            pos_tag = pos[0][1]
            # 조사(J), 접속사, 일부 부사만
            if pos_tag.startswith('J') or pos_tag in ['MAJ', 'MAC']:
                improved_v4.append(word)
    except:
        pass

print(f"\n✅ V4: 조사/접속사만 (가장 보수적)")
print(f"   원본: {len(general_stopwords)}개 → 개선: {len(improved_v4)}개")

# =========================================================
# 6. 파일 저장
# =========================================================
output_dir = "improved_general_stopwords"
os.makedirs(output_dir, exist_ok=True)

# V1 저장
with open(f'{output_dir}/general_v1_no_legal.txt', 'w', encoding='utf-8') as f:
    for word in improved_v1:
        f.write(word + '\n')

# V2 저장  
with open(f'{output_dir}/general_v2_no_legal_no_sound.txt', 'w', encoding='utf-8') as f:
    for word in improved_v2:
        f.write(word + '\n')

# V3 저장
with open(f'{output_dir}/general_v3_conservative.txt', 'w', encoding='utf-8') as f:
    for word in improved_v3:
        f.write(word + '\n')

# V4 저장
with open(f'{output_dir}/general_v4_particles_only.txt', 'w', encoding='utf-8') as f:
    for word in improved_v4:
        f.write(word + '\n')

# 제거된 법률 용어 목록
with open(f'{output_dir}/removed_legal_terms.txt', 'w', encoding='utf-8') as f:
    f.write("# 원본 General 리스트에서 제거해야 할 법률 핵심 용어\n\n")
    for word in found_legal_terms:
        f.write(word + '\n')

print(f"\n💾 저장 완료:")
print(f"   - {output_dir}/general_v1_no_legal.txt ({len(improved_v1)}개)")
print(f"   - {output_dir}/general_v2_no_legal_no_sound.txt ({len(improved_v2)}개)")
print(f"   - {output_dir}/general_v3_conservative.txt ({len(improved_v3)}개)")
print(f"   - {output_dir}/general_v4_particles_only.txt ({len(improved_v4)}개)")
print(f"   - {output_dir}/removed_legal_terms.txt")

# =========================================================
# 7. 요약 및 권장사항
# =========================================================
print(f"\n{'='*70}")
print(f"  📋 요약 및 권장사항")
print(f"{'='*70}")

print(f"\n🚨 핵심 문제:")
print(f"   원본 General 리스트가 '조', '항', '호', '목', '법'을 포함하고 있습니다.")
print(f"   이들은 법률 문서 구조를 나타내는 핵심 용어로, 제거 시 분류 성능 하락!")

print(f"\n💡 권장 순서:")
print(f"   1️⃣  먼저 V1 테스트 (법률 용어만 제거)")
print(f"      → 예상: F1 = 0.8750+ (현재 0.8741보다 상승)")
print(f"\n   2️⃣  V1이 성공하면 V2 테스트 (+ 의성어 제거)")
print(f"      → 예상: F1 유지 또는 소폭 상승")
print(f"\n   3️⃣  V3/V4는 보수적 옵션")
print(f"      → V4가 가장 안전하지만 단어 수 매우 적음")

print(f"\n🎯 예상 결과:")
print(f"   - General V1 (법률 용어 제거): F1 = 0.8750-0.8760")
print(f"   - General V2 (+ 의성어 제거):  F1 = 0.8745-0.8755")
print(f"   - General V4 (조사만):         F1 = 0.8720-0.8730")

print(f"\n다음 명령어로 바로 테스트:")
print(f"   python benchmark_general_improved.py")
print(f"\n{'='*70}\n")

  General Stopwords Analysis

총 600개 불용어

🚨 발견된 법률 핵심 용어 (5개):
   ['조', '항', '호', '목', '법']

   ❌ 이 단어들은 절대 제거하면 안 됩니다!

💭 의성어/의태어 (36개):
   ['까악', '꽈당', '끙끙', '끼익', '둥둥', '딩동', '뚝뚝', '붕붕', '삐걱', '솨', '쉿', '엉엉', '와르르', '와아', '윙윙', '졸졸', '좍좍', '주룩주룩', '콸콸', '쾅쾅', '쿵', '탕탕', '툭', '퉤', '팍', '퍽', '펄렁', '허걱', '허허', '헉', '헉헉', '헐떡헐떡', '휘익', '휴', '흐흐', '흥']
   ℹ️  법률 문서에서 거의 등장하지 않음 (제거해도 무방)

⏳ Mecab 품사 분석 중...

📊 품사 분포 (상위 10개):
   NNG: 161개
   MAG: 98개
   VV: 77개
   NP: 37개
   MM: 33개
   VV+EC: 32개
   IC: 23개
   MAJ: 19개
   VV+ETM: 18개
   NR: 17개

🔍 명사로 태깅된 단어 (243개):
   ['각자', '각종', '개의치않고', '거니와', '것', '것과 같이', '것들', '견지에서', '결과에 이르다', '결국', '결론을 낼 수 있다', '고려하면', '고로', '공동으로', '과', '관계가 있다', '관련이 있다', '구', '구체적으로', '구토하다', '그들', '그때', '그에 따르는', '근거로', '근거하여', '기점으로', '기준으로', '기타', '까닭으로', '나']...

⚠️  문제 가능성 있는 단어 (17개):
   ['결국', '기타', '년', '등', '등등', '시각', '시간', '월', '일', '전부', '전자', '전후', '조', '항', '호', '목', '법']

  개선안 생성

✅ V1: 법률 핵심 용어 제거
   원본: 600개 → 개선: 595개
   제거된 단어: ['조', '항'

In [3]:
import os
import glob
import json
import pandas as pd
import numpy as np
import time
import warnings
from tqdm import tqdm
from konlpy.tag import Mecab
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.feature_extraction.text")

# =========================================================
# 설정
# =========================================================
DATA_DIRS = ["data/대법원_판례", "data/하위법원_판례"] 
OUTPUT_DIR = "benchmark_general_improved"
TOP_N_CLASSES = 10
NUM_ITERATIONS = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f">>> 결과 저장 폴더: {OUTPUT_DIR}")

mecab = Mecab()

def pre_tokenize(text):
    if not text: return ""
    return " ".join([word for word, pos in mecab.pos(text) if pos.startswith('N')])

def load_and_preprocess_data(data_dirs_list, top_n=None):
    all_files = []
    for d in data_dirs_list:
        files = glob.glob(os.path.join(d, "**", "*.json"), recursive=True)
        all_files.extend(files)
    
    texts, labels = [], []
    
    print(f">>> 총 {len(all_files)}개 JSON 파일 발견")
    for file_path in tqdm(all_files, desc="JSON Loading"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                content = data.get('판례내용', "").replace("<br/>", " ").replace("\n", " ")
                label = data.get('사건명', "")
                if content and label:
                    texts.append(content)
                    labels.append(label)
        except: 
            continue
    
    df = pd.DataFrame({'text': texts, 'label': labels})
    print(f">>> 로딩 완료: {len(df)}개 문서")
    
    if top_n:
        target_classes = df['label'].value_counts().head(top_n).index.tolist()
        df = df[df['label'].isin(target_classes)].copy()
        print(f">>> Top {top_n} 클래스 필터링: {len(df)}개 문서")
    
    tqdm.pandas(desc="Pre-tokenizing (Mecab)")
    df['tokenized_text'] = df['text'].progress_apply(pre_tokenize)
    
    return train_test_split(
        df['tokenized_text'], df['label'], 
        test_size=0.2, random_state=42, stratify=df['label']
    )

def load_stopwords(filepath):
    if not os.path.exists(filepath):
        return None
    with open(filepath, 'r', encoding='utf-8') as f:
        return set([line.strip() for line in f if line.strip()])

print(f"\n{'='*70}")
print(f"  Improved General Stopwords - Benchmark Test")
print(f"{'='*70}\n")

print(f">>> 1. 데이터셋 로딩 및 사전 토큰화...")
X_train, X_test, y_train, y_test = load_and_preprocess_data(DATA_DIRS, TOP_N_CLASSES)

# =========================================================
# 불용어 준비
# =========================================================
print(f"\n>>> 2. 불용어 리스트 로드 중...")

# 먼저 analyze_general_stopwords.py 실행 필요 여부 확인
IMPROVED_DIR = "improved_general_stopwords"
if not os.path.exists(IMPROVED_DIR):
    print(f"\n⚠️  {IMPROVED_DIR} 폴더가 없습니다!")
    print(f"먼저 다음 명령어를 실행하세요:")
    print(f"   python analyze_general_stopwords.py")
    print(f"\n수동으로 개선된 버전 생성 중...")
    
    # 원본 로드
    general_original = load_stopwords('stopwords.txt')
    
    # 법률 핵심 용어 제거
    LEGAL_TERMS_TO_REMOVE = {'조', '항', '호', '목', '법'}
    general_v1 = general_original - LEGAL_TERMS_TO_REMOVE
    
    # 저장
    os.makedirs(IMPROVED_DIR, exist_ok=True)
    with open(f'{IMPROVED_DIR}/general_v1_no_legal.txt', 'w', encoding='utf-8') as f:
        for word in sorted(general_v1):
            f.write(word + '\n')
    
    print(f"✅ V1 생성 완료: {len(general_v1)}개 (법률 용어 제거)")

# 불용어 로드
general_original = load_stopwords('stopwords.txt')
general_v1 = load_stopwords(f'{IMPROVED_DIR}/general_v1_no_legal.txt')
general_v2 = load_stopwords(f'{IMPROVED_DIR}/general_v2_no_legal_no_sound.txt')
general_v3 = load_stopwords(f'{IMPROVED_DIR}/general_v3_conservative.txt')
general_v4 = load_stopwords(f'{IMPROVED_DIR}/general_v4_particles_only.txt')

# Conservative (비교용)
conservative = load_stopwords('improved_stopwords/V6_Conservative.txt')

# =========================================================
# 실험 시나리오
# =========================================================
stopword_sets = {
    "0. Baseline (None)": set(),
    "1. General Original": general_original,
}

if general_v1:
    stopword_sets["2. General V1 (No Legal)"] = general_v1
if general_v2:
    stopword_sets["3. General V2 (No Legal+Sound)"] = general_v2
if general_v3:
    stopword_sets["4. General V3 (Conservative)"] = general_v3
if general_v4:
    stopword_sets["5. General V4 (Particles Only)"] = general_v4
if conservative:
    stopword_sets["[Ref] Conservative"] = conservative

# 하이브리드: V1 + Conservative
if general_v1 and conservative:
    union_v1_conservative = general_v1.union(conservative)
    stopword_sets["6. V1 ∪ Conservative"] = union_v1_conservative

print(f"\n>>> 총 {len(stopword_sets)}개 실험 시나리오")
for name, sset in stopword_sets.items():
    print(f"   {name}: {len(sset)}개")

# =========================================================
# 벤치마크
# =========================================================
print(f"\n{'='*110}")
print(f"  분류 성능 벤치마크 (Top {TOP_N_CLASSES} 클래스, {NUM_ITERATIONS}회 반복)")
print(f"{'='*110}\n")

results = []
print("-" * 110)
print(f"{'Experiment':<40} | {'Acc':<8} | {'F1':<8} | {'Time':<12} | {'Count'}")
print("-" * 110)

for name, stop_set in stopword_sets.items():
    stop_list = list(stop_set) if stop_set else None
    times, accuracies, f1s = [], [], []
    
    for _ in range(NUM_ITERATIONS):
        pipeline = Pipeline([
            ('vect', TfidfVectorizer(
                stop_words=stop_list, 
                max_features=5000,
                tokenizer=None,
                token_pattern=r"(?u)\b\w\w+\b"
            )),
            ('clf', LogisticRegression(C=5.0, max_iter=1000, n_jobs=-1)) 
        ])
        
        start_time = time.time()
        try:
            pipeline.fit(X_train, y_train)
            pred = pipeline.predict(X_test)
            elapsed = time.time() - start_time
            
            times.append(elapsed)
            accuracies.append(accuracy_score(y_test, pred))
            f1s.append(f1_score(y_test, pred, average='weighted'))
        except Exception as e:
            print(f"{name:<40} | Error: {e}")
            break
    
    if times:
        avg_acc = np.mean(accuracies)
        avg_f1 = np.mean(f1s)
        avg_time = np.mean(times)
        std_time = np.std(times)
        
        # 성능 개선 표시
        status = ""
        if 'original' not in name.lower() and avg_f1 > 0.8741:
            status = "⭐ IMPROVED!"
        
        print(f"{name:<40} | {avg_acc:.4f}   | {avg_f1:.4f}   | {avg_time:.2f}s ±{std_time:.2f} | {len(stop_set):<6} {status}")
        
        results.append({
            'Experiment': name,
            'Count': len(stop_set),
            'Accuracy': avg_acc,
            'F1_Score': avg_f1,
            'Avg_Time': avg_time,
            'Std_Time': std_time
        })

print("-" * 110)

# =========================================================
# 결과 분석
# =========================================================
if results:
    final_df = pd.DataFrame(results).sort_values(
        by=['F1_Score', 'Accuracy'], 
        ascending=[False, False]
    ).round(4)
    
    final_df.insert(0, 'Rank', range(1, len(final_df) + 1))
    
    print("\n" + "="*90)
    print(f"  📊 개선된 General 불용어 성능 순위")
    print("="*90)
    print(final_df.to_string(index=False))
    print("="*90)
    
    # CSV 저장
    csv_path = os.path.join(OUTPUT_DIR, f"benchmark_general_improved_top{TOP_N_CLASSES}.csv")
    final_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 결과 저장: {csv_path}")
    
    # 핵심 비교
    print(f"\n{'='*90}")
    print(f"  🎯 핵심 비교: Original vs V1 (법률 용어 제거)")
    print(f"{'='*90}")
    
    original = final_df[final_df['Experiment'] == '1. General Original']
    v1 = final_df[final_df['Experiment'] == '2. General V1 (No Legal)']
    
    if not original.empty and not v1.empty:
        orig_f1 = original['F1_Score'].values[0]
        v1_f1 = v1['F1_Score'].values[0]
        
        diff = (v1_f1 - orig_f1) * 100
        
        print(f"\nOriginal (600개, 포함: 조,항,호,목,법):")
        print(f"  F1 = {orig_f1:.4f}")
        
        print(f"\nV1 ({len(general_v1)}개, 법률 용어 제거):")
        print(f"  F1 = {v1_f1:.4f}")
        
        print(f"\n변화: {diff:+.2f}% ({diff/100*orig_f1:+.4f})")
        
        if v1_f1 > orig_f1:
            print(f"\n🎉 성공! 법률 용어 제거로 성능 향상!")
        elif abs(v1_f1 - orig_f1) < 0.0005:
            print(f"\n✅ 성능 유지 (거의 동일)")
        else:
            print(f"\n⚠️  약간 하락했지만 예상 범위 내")
    
    # 최고 성능
    best = final_df.iloc[0]
    print(f"\n{'='*90}")
    print(f"  🏆 최종 권장")
    print(f"{'='*90}")
    print(f"\n최고 F1 성능: {best['Experiment']}")
    print(f"  - F1: {best['F1_Score']:.4f}")
    print(f"  - Count: {best['Count']}개")
    print(f"  - Time: {best['Avg_Time']:.2f}s")
    
    # 제거된 단어의 영향
    if general_v1:
        removed = general_original - general_v1
        print(f"\n제거된 법률 용어: {sorted(removed)}")
    
    print(f"\n{'='*90}\n")

print(f"✅ 개선된 General 불용어 테스트 완료!")
print(f"결과 폴더: {OUTPUT_DIR}/")

>>> 결과 저장 폴더: benchmark_general_improved

  Improved General Stopwords - Benchmark Test

>>> 1. 데이터셋 로딩 및 사전 토큰화...
>>> 총 33230개 JSON 파일 발견


JSON Loading: 100%|██████████| 33230/33230 [00:12<00:00, 2715.95it/s]


>>> 로딩 완료: 33230개 문서
>>> Top 10 클래스 필터링: 4763개 문서


Pre-tokenizing (Mecab): 100%|██████████| 4763/4763 [00:17<00:00, 264.68it/s]



>>> 2. 불용어 리스트 로드 중...

>>> 총 8개 실험 시나리오
   0. Baseline (None): 0개
   1. General Original: 600개
   2. General V1 (No Legal): 595개
   3. General V2 (No Legal+Sound): 559개
   4. General V3 (Conservative): 547개
   5. General V4 (Particles Only): 37개
   [Ref] Conservative: 75개
   6. V1 ∪ Conservative: 630개

  분류 성능 벤치마크 (Top 10 클래스, 5회 반복)

--------------------------------------------------------------------------------------------------------------
Experiment                               | Acc      | F1       | Time         | Count
--------------------------------------------------------------------------------------------------------------
0. Baseline (None)                       | 0.8730   | 0.8708   | 3.54s ±0.55 | 0      
1. General Original                      | 0.8762   | 0.8741   | 3.47s ±0.07 | 600    
2. General V1 (No Legal)                 | 0.8762   | 0.8741   | 3.47s ±0.07 | 595    
3. General V2 (No Legal+Sound)           | 0.8762   | 0.8741   | 3.48s ±0.03 | 559    
4. G